In [1]:
import pandas as pd
import re
import pymorphy3
import json

needed_columns = ['product_id', 'shop_name', 'category', 'name_product', 'url_product']
df = pd.read_excel('farmers_sku.xlsx', usecols=needed_columns)

df = df.dropna(subset=['name_product', 'url_product'])
df['name_product'] = df['name_product'].str.lower().str.strip()
df = df.drop_duplicates(subset=['name_product', 'shop_name'])

bad_categories = [
    'сладости', 'напитки', 'хлеб и выпечка', 
    'яйца и молочные продукты', 'мед', 'мёд', 'сыры', 'колбасы'
]
df = df[~df['category'].str.lower().isin(bad_categories)]

stop_words = (
    'драже|конфет|печенье|торт|пирожн|лимонад|готовы|полуфабрикат|'
    'варенье|джем|мармелад|зефир|шоколад|чипсы|пельмени|вареники|'
    'блинчики|сырники|чай|сбор|травян|набор|подарок|корзин|'
    'пастила|цукат|фрипс|паштет|колбас|сосиск|мюсли|котлет|'
    'фрикадельк|голубц|тефтел|манты|'
    'масло|сметана|творог|кефир|молоко|яйцо|сыр|'
    'мука|крупа|рис|гречка|мед|мёд|сахар|соль|рулет|блины'
)
df = df[~df['name_product'].str.contains(stop_words, case=False, na=False)].copy()
morph = pymorphy3.MorphAnalyzer()

def extract_ingredients(row):
    try:
        name = str(row['name_product'])
        words = re.findall(r'[а-яёa-z]+', name) 
        
        main_ingredient = None
        attributes = []
        ignore_words = {'кг', 'г', 'гр', 'л', 'мл', 'шт', 'упаковка', 'вес', 'белополе'}
            
        for word in words:
            if word in ignore_words: continue 
                
            parsed = morph.parse(word)[0]
            if any(tag in parsed.tag for tag in {'PREP', 'CONJ', 'PRCL', 'INTJ'}): continue
                
            normal_form = parsed.normal_form
            
            exceptions = {
                'белые': 'гриб', 'сливовыя': 'сливовый', 'огурчик': 'огурец', 
                'томат': 'помидор', 'редиска': 'редис', 'свёкла': 'свекла',
                'чипc': 'чипсы', 'юрчать': 'юрча'
            }
            normal_form = exceptions.get(normal_form, normal_form)
            
            if not main_ingredient and 'NOUN' in parsed.tag:
                main_ingredient = normal_form
            else:
                if normal_form != main_ingredient:
                    attributes.append(normal_form)

        if not main_ingredient:
            main_ingredient = morph.parse(words[0])[0].normal_form if words else 'продукт'
                
        return pd.Series(
            [main_ingredient, json.dumps(attributes, ensure_ascii=False)], 
            index=['main_ingredient', 'attributes']
        )
    except Exception:
        return pd.Series(['ошибка_парсинга', '[]'], index=['main_ingredient', 'attributes'])

df[['main_ingredient', 'attributes']] = df.apply(extract_ingredients, axis=1)
df = df[df['main_ingredient'] != 'ошибка_парсинга']


seasonal_map = {
    'черемша':    [0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0],
    'крапива':    [0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0],
    'щавель':     [0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0],
    'шпинат':     [0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0],
    'редис':      [0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0],
    'спаржа':     [0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0],
    'сморчок':    [0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0], 
    'корюшка':    [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],

    'огурец':     [0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0], 
    'помидор':    [0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0],
    'кабачок':    [0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0],
    'цукиня':     [0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0], 
    'патиссон':   [0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0],
    'баклажан':   [0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0],
    'перец':      [0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0],
    'кукуруза':   [0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0],
    'клубника':   [0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0],
    'земляника':  [0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0],
    'черешня':    [0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0],
    'малина':     [0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0],
    'смородина':  [0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0],
    'вишня':      [0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0],
    'ежевика':    [0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0],
    'крыжовник':  [0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0],
    'черника':    [0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0],
    'голубика':   [0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0],
    'абрикос':    [0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0],
    'персик':     [0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0],

    'картофель':  [1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1], 
    'морковь':    [1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1],
    'свекла':     [1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1],
    'лук':        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
    'чеснок':     [1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1],
    'капуста':    [1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1],
    'тыква':      [1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1], 
    'репка':      [1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1],
    'редька':     [1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1],
    'яблоко':     [1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1], 
    'груша':      [1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1],
    'слива':      [0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0],
    'арбуз':      [0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0],
    'дыня':       [0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0],
    'облепиха':   [0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0],
    'клюква':     [1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1],
    'брусника':   [1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1],
    'шиповник':   [1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1],
    'боярышник':  [1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1],

    'лисичка':    [0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0],
    'маслёнок':   [0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0],
    'моховик':    [0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0],
    'подосиновик':[0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0],
    'рыжик':      [0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0],
    'опёнок':     [0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0],
    'груздь':     [0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0],
    'шампиньон':  [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 
    'гриб':       [0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0], 

    'утка':       [0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0],
    'уточка':     [0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0],
    'гусь':       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
    'оленина':    [1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1],
    'лосятина':   [1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1],
    'заяц':       [1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1],
    'налим':      [1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
    
    'укроп':      [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
    'петрушка':   [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
    'кинза':      [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
    'базилик':    [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
    'руккола':    [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
    'микрозелень':[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
}

def apply_seasonality(row):
    ing = str(row['main_ingredient'])
    attr_str = str(row['attributes']).lower()
    
    months = seasonal_map.get(ing, [1] * 12)
    
    storage_words = ['заморозить', 'сушен', 'засуш', 'мороз', 'консерв', 'солен', 'мариновать', 'квашен', 'вялен']
    if any(word in attr_str for word in storage_words):
        months = [1] * 12
        
    month_names = [f'month_{i}' for i in range(1, 13)]
    return pd.Series(months, index=month_names)

month_cols = [f'month_{i}' for i in range(1, 13)]
df[month_cols] = df.apply(apply_seasonality, axis=1)

premium_items = [
    'оленина', 'лосятина', 'косуля', 'кабан', 'заяц', 
    'перепел', 'перепёлка', 'цесарка', 'ягнёнок', 'козлёнок', 'козлятина',
    
    'стейк', 'рибай', 'стриплойн', 'ростбиф', 'карпаччо', 
    'хамон', 'прошутто', 'брезаола', 'балык', 'суджук', 
    'рийет', 'панчетта', 'пальчетто', 'коппа', 'мортаделла', 
    'лардо', 'фуэт', 'lomo', 'петто',

    'икра', 'осетр', 'осётр', 'осетрина', 'стерлядь', 
    'муксун', 'омуль', 'нельма', 'пелядь', 'хариус', 'чавыча', 'нерка',
    'лосось', 'сёмга', 'форель', 'тунец', 'сибас', 'угорь',

    'краб', 'крабик', 'гребешок', 'лангустин', 'устрица', 
    'жилардо', 'хасанский', 'мидия', 'каракатица', 
    'осьминог', 'осьминожка', 'ботан', 

    'трюфель', 'сморчок', 'лисичка', 

    'буррата', 'страчателла', 'камамбер', 'бри', 'горгонзола',
    'рокфор', 'пармезан', 'рикотта', 'халуми',

    'спаржа', 'артишок', 'микрозелень', 'морошка', 
    'улитка', 'гинкго', 'тапенада', 'песто', 'кедрокофе'
]
df['is_premium'] = df['main_ingredient'].isin(premium_items).astype(int)

df['months_available'] = df[month_cols].sum(axis=1)
df['rarity_score'] = 13 - df['months_available']
df = df.drop(columns=['months_available'])

df.to_csv('HACKATHON_FINAL_DB.csv', index=False, encoding='utf-8-sig')
